# PDF / DOCX / API Data Processing Assignment

This notebook implements Tasks 1–9. It expects the following sample files to exist
in the working directory (adjust paths in the first cell of each task as needed):

- `sample.pdf` — a multi-page PDF (Tasks 1–3)
- `sample.docx` — a .docx file with varied paragraph styles (Task 4)
- `sample_with_table.docx` — a .docx file containing at least one table (Task 5)
- `document_a.docx`, `document_b.docx` — two .docx files with headings + Normal paragraphs (Task 6)

Tasks 7–9 use the public JSONPlaceholder API and require internet access.

Run the setup cell below first to install/import required packages.

In [1]:
# Setup: install required packages (uncomment if needed)
# !pip install PyPDF2 pdfplumber python-docx pandas nltk requests --quiet

import re
import csv
import json
import math
import string
from collections import Counter, defaultdict

import requests
import pandas as pd

try:
    import nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)
    NLTK_STOPWORDS = set(stopwords.words('english'))
except Exception:
    # Fallback list if nltk / corpus download is unavailable
    NLTK_STOPWORDS = {
        'i','me','my','myself','we','our','ours','ourselves','you',"you're","you've",
        'your','yours','yourself','yourselves','he','him','his','himself','she',"she's",
        'her','hers','herself','it',"it's",'its','itself','they','them','their','theirs',
        'themselves','what','which','who','whom','this','that',"that'll",'these','those',
        'am','is','are','was','were','be','been','being','have','has','had','having','do',
        'does','did','doing','a','an','the','and','but','if','or','because','as','until',
        'while','of','at','by','for','with','about','against','between','into','through',
        'during','before','after','above','below','to','from','up','down','in','out','on',
        'off','over','under','again','further','then','once'
    }

BASIC_STOPWORDS = {
    'the','a','an','and','or','but','is','are','was','were','in','on','at','to','for',
    'of','with','as','by','that','this','it','be','from','has','have','had','not','but',
    'they','you','we','he','she','his','her','their','its','which','who','whom','will'
}
print("Setup complete.")

Setup complete.


---
## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

In [3]:
# Task 1(a): Extract text from each page individually
!pip install pdfplumber --quiet

import pdfplumber

PDF_PATH = "sample.pdf"

def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages.append(text)
    return pages

pages_list = extract_pages(PDF_PATH)
print(f"Total number of pages extracted: {len(pages_list)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 104.0 MB/s eta 0:00:00
Total number of pages extracted: 6


In [4]:
# Task 1(b): Merge pages, clean, tokenize
full_text = " ".join(pages_list)

cleaned = re.sub(r'[\d]+', '', full_text)                 # remove digits
cleaned = re.sub(f"[{re.escape(string.punctuation)}]", '', cleaned)  # remove punctuation
cleaned = cleaned.lower()

tokens = cleaned.split()
print(f"Total token count: {len(tokens)}")

Total token count: 599


In [5]:
# Task 1(c): Word frequency dictionary with stopword removal
stopwords_set = BASIC_STOPWORDS | {
    'i','you','he','she','we','they','them','him','her','so','if','than','then',
    'when','where','what','how','why','can','could','would','should','there'
}
assert len(stopwords_set) >= 20, "Need at least 20 stopwords"

filtered_tokens = [t for t in tokens if t not in stopwords_set and len(t) > 0]
freq = Counter(filtered_tokens)

print("Top 10 most frequent words:")
for word, count in freq.most_common(10):
    print(f"{word}: {count}")

Top 10 most frequent words:
energy: 16
wind: 13
solar: 11
renewable: 10
more: 6
storage: 6
electricity: 5
farms: 5
sources: 4
panels: 4


In [6]:
# Task 1(d): Keyword search across pages, sentence-level
keyword = input("Enter a keyword to search for: ").strip().lower()

found_any = False
for page_num, page_text in enumerate(pages_list, start=1):
    sentences = re.split(r'(?<=[.!?])\s+', page_text)
    for sentence in sentences:
        if keyword and keyword in sentence.lower():
            print(f"[Page {page_num}] {sentence.strip()}")
            found_any = True

if not found_any:
    print(f"No sentences found containing the keyword '{keyword}'.")

Enter a keyword to search for: sources
[Page 1] Introduction to Renewable Energy
Renewable energy comes from natural sources that replenish themselves faster than we use them,
such as sunlight, wind, rain, tides, waves, and geothermal heat.
[Page 1] Many countries have set ambitious targets to reduce carbon emissions and transition
their power grids toward cleaner sources.
[Page 1] Energy storage, especially battery technology, remains a key challenge for making renewable
sources reliable around the clock.
[Page 6] Future Outlook
Looking ahead, analysts project that renewable energy sources could supply the majority of global
electricity generation within the next two decades.
[Page 6] For additional resources, see
https://www.futureenergy.example.net.


---
## Task 2: Structured Page Metadata Extraction and CSV Export

In [7]:
# Task 2(a): Build pages_data list of dicts
PDF_PATH_2 = "sample.pdf"  # must have >= 5 pages

pages_data = []
with pdfplumber.open(PDF_PATH_2) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        raw_text = page.extract_text() or ""
        word_count = len(raw_text.split())
        char_count = len(raw_text.replace(" ", "").replace("\n", ""))
        pages_data.append({
            "page_number": i,
            "raw_text": raw_text,
            "word_count": word_count,
            "char_count": char_count
        })

print(f"Total pages: {len(pages_data)}")
for entry in pages_data[:2]:
    print(entry)

Total pages: 6
{'page_number': 1, 'raw_text': 'Introduction to Renewable Energy\nRenewable energy comes from natural sources that replenish themselves faster than we use them,\nsuch as sunlight, wind, rain, tides, waves, and geothermal heat. Over the past decade, the cost of solar\npanels and wind turbines has dropped dramatically, making renewable energy increasingly competitive\nwith fossil fuels. Many countries have set ambitious targets to reduce carbon emissions and transition\ntheir power grids toward cleaner sources. Visit https://www.example.com/renewables for more\ninformation. Solar power alone accounted for a significant share of new electricity capacity added\nworldwide in 2023. Wind energy also continues to grow, particularly offshore wind farms in Europe and\nAsia. Energy storage, especially battery technology, remains a key challenge for making renewable\nsources reliable around the clock.', 'word_count': 121, 'char_count': 741}
{'page_number': 2, 'raw_text': 'Solar Powe

In [8]:
# Task 2(b): Highest / lowest non-zero word count / average
non_zero_pages = [p for p in pages_data if p["word_count"] > 0]

if non_zero_pages:
    highest_page = max(pages_data, key=lambda p: p["word_count"])
    lowest_page = min(non_zero_pages, key=lambda p: p["word_count"])
    avg_word_count = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2)

    print(f"Highest word count -> Page {highest_page['page_number']} "
          f"({highest_page['word_count']} words)")
    print(f"Content:\n{highest_page['raw_text']}\n")

    print(f"Lowest non-zero word count -> Page {lowest_page['page_number']} "
          f"({lowest_page['word_count']} words)")

    print(f"Average word count across all pages: {avg_word_count}")
else:
    print("All pages returned zero words; cannot compute lowest non-zero word count.")

Highest word count -> Page 1 (121 words)
Content:
Introduction to Renewable Energy
Renewable energy comes from natural sources that replenish themselves faster than we use them,
such as sunlight, wind, rain, tides, waves, and geothermal heat. Over the past decade, the cost of solar
panels and wind turbines has dropped dramatically, making renewable energy increasingly competitive
with fossil fuels. Many countries have set ambitious targets to reduce carbon emissions and transition
their power grids toward cleaner sources. Visit https://www.example.com/renewables for more
information. Solar power alone accounted for a significant share of new electricity capacity added
worldwide in 2023. Wind energy also continues to grow, particularly offshore wind farms in Europe and
Asia. Energy storage, especially battery technology, remains a key challenge for making renewable
sources reliable around the clock.

Lowest non-zero word count -> Page 6 (87 words)
Average word count across all pages: 10

In [9]:
# Task 2(c): Custom exception for phrase search
class PhraseNotFoundError(Exception):
    pass

def search_phrase(pages_data, phrase):
    phrase_lower = phrase.lower()
    results = []
    for p in pages_data:
        count = p["raw_text"].lower().count(phrase_lower)
        if count > 0:
            results.append((p["page_number"], count))
    if not results:
        raise PhraseNotFoundError(f"Phrase '{phrase}' was not found in any page.")
    return results

search_term = input("Enter a search phrase: ").strip()

try:
    matches = search_phrase(pages_data, search_term)
    for page_number, count in matches:
        print(f"Page {page_number}: {count} occurrence(s)")
except PhraseNotFoundError as e:
    print(f"Caught PhraseNotFoundError: {e}")

Enter a search phrase: helloo
Caught PhraseNotFoundError: Phrase 'helloo' was not found in any page.


In [10]:
# Task 2(d): Write and re-read pdf_summary.csv using the csv module
csv_filename = "pdf_summary.csv"

with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

print(f"Wrote {csv_filename}. Re-reading contents:")
with open(csv_filename, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

Wrote pdf_summary.csv. Re-reading contents:
['page_number', 'word_count', 'char_count']
['1', '121', '741']
['2', '109', '660']
['3', '105', '627']
['4', '92', '579']
['5', '98', '639']
['6', '87', '593']


---
## Task 3: PDF Text Preprocessing Pipeline for NLP

In [11]:
# Task 3(a): Extract text and clean it in order
PDF_PATH_3 = "sample.pdf"

with pdfplumber.open(PDF_PATH_3) as pdf:
    raw_full_text = " ".join((page.extract_text() or "") for page in pdf.pages)

char_count_before = len(raw_full_text)

step_text = re.sub(r'https?://\S+|www\.\S+', '', raw_full_text)          # (i) remove URLs
step_text = re.sub(r'\d+', '', step_text)                                 # (ii) remove digits
step_text = re.sub(r'[^A-Za-z\s.]', '', step_text)                        # (iii) keep letters/spaces/periods
step_text = re.sub(r'\s+', ' ', step_text).strip()                        # (iv) collapse spaces

char_count_after = len(step_text)

print(f"Character count before cleaning: {char_count_before}")
print(f"Character count after cleaning: {char_count_after}")

Character count before cleaning: 4450
Character count after cleaning: 4249


In [12]:
# Task 3(b): Lowercase, split into sentences, strip, remove empties
lower_text = step_text.lower()
raw_sentences = re.split(r'[.!?]', lower_text)

sentence_count_before = len(raw_sentences)

sentences = [s.strip() for s in raw_sentences]
sentences = [s for s in sentences if s != ""]

sentence_count_after = len(sentences)

print(f"Sentence count before removing empties: {sentence_count_before}")
print(f"Sentence count after removing empties: {sentence_count_after}")

Sentence count before removing empties: 34
Sentence count after removing empties: 33


In [13]:
# Task 3(c): Tokenize, remove stopwords, filter short sentences
sentence_count_before_filter = len(sentences)

filtered_sentences = []
for sent in sentences:
    tokens = sent.split()
    tokens_no_stop = [t for t in tokens if t not in NLTK_STOPWORDS]
    if len(tokens_no_stop) >= 5:
        filtered_sentences.append(" ".join(tokens_no_stop))

sentence_count_after_filter = len(filtered_sentences)

print(f"Sentence count before stopword-length filter: {sentence_count_before_filter}")
print(f"Sentence count after stopword-length filter: {sentence_count_after_filter}")
print("Sample sentences:")
for s in filtered_sentences[:3]:
    print(f"- {s}")

Sentence count before stopword-length filter: 33
Sentence count after stopword-length filter: 32
Sample sentences:
- introduction renewable energy renewable energy comes natural sources replenish faster use sunlight wind rain tides waves geothermal heat
- past decade cost solar panels wind turbines dropped dramatically making renewable energy increasingly competitive fossil fuels
- many countries set ambitious targets reduce carbon emissions transition power grids toward cleaner sources


In [14]:
# Task 3(d): Write to file, re-read, and report stats
output_filename = "nlp_ready_output.txt"

with open(output_filename, "w", encoding="utf-8") as f:
    for sent in filtered_sentences:
        f.write(sent + "\n")

with open(output_filename, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f.readlines()]

total_lines = len(lines)
longest_sentence = max(lines, key=len) if lines else ""
shortest_sentence = min(lines, key=len) if lines else ""

print(f"Total lines written: {total_lines}")
print(f"Longest sentence ({len(longest_sentence)} chars): {longest_sentence}")
print(f"Shortest sentence ({len(shortest_sentence)} chars): {shortest_sentence}")

Total lines written: 32
Longest sentence (171 chars): efficiency improvements continue steadily laboratory cells exceeding percent efficiency concentrated sunlight though commercial panels typically operate percent efficiency
Shortest sentence (42 chars): read technical specifications case studies


---
## Task 4: DOCX Paragraph Classification and Statistical Analysis

In [16]:
# Task 4(a): Load paragraphs and styles
!pip install python-docx --quiet

from docx import Document

DOCX_PATH = "sample.docx"

doc = Document(DOCX_PATH)

paragraph_records = []
for p in doc.paragraphs:
    paragraph_records.append({"text": p.text, "style": p.style.name})

unique_styles = sorted(set(r["style"] for r in paragraph_records))

print(f"Total paragraph count: {len(paragraph_records)}")
print(f"Unique style names found: {unique_styles}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.0 MB/s eta 0:00:00
Total paragraph count: 16
Unique style names found: ['Heading 1', 'Heading 2', 'Normal', 'Title']


In [17]:
# Task 4(b): Classify into Headings / Body / Other-Empty
headings, body, other_empty = [], [], []

for r in paragraph_records:
    style = r["style"]
    text = r["text"].strip()
    if style.startswith("Heading"):
        headings.append(r)
    elif style == "Normal" or "Body" in style:
        body.append(r)
    else:
        other_empty.append(r)

print(f"Headings: {len(headings)}")
print(f"Body: {len(body)}")
print(f"Other/Empty: {len(other_empty)}")

Headings: 5
Body: 10
Other/Empty: 1


In [18]:
# Task 4(c): Stats on qualifying Body paragraphs (> 5 words)
qualifying = []
for r in body:
    word_count = len(r["text"].split())
    if word_count > 5:
        char_count = len(r["text"].replace(" ", ""))
        qualifying.append({**r, "word_count": word_count, "char_count": char_count})

if qualifying:
    highest_wc = max(qualifying, key=lambda r: r["word_count"])
    lowest_wc = min(qualifying, key=lambda r: r["word_count"])
    mean_wc = round(sum(r["word_count"] for r in qualifying) / len(qualifying), 2)

    print(f"Highest word count paragraph ({highest_wc['word_count']} words): {highest_wc['text']}")
    print(f"Lowest word count (>5) paragraph ({lowest_wc['word_count']} words): {lowest_wc['text']}")
    print(f"Mean word count across qualifying body paragraphs: {mean_wc}")
else:
    print("No body paragraphs with more than 5 words were found.")

Highest word count paragraph (44 words): Your first task is to set up your laptop and accounts. IT will provide you with login credentials for email, chat, and the internal wiki on your first day. Make sure to change your temporary password immediately after logging in for the first time.
Lowest word count (>5) paragraph (19 words): Please read each section carefully and reach out to your manager if you have any questions along the way.
Mean word count across qualifying body paragraphs: 33.83


In [19]:
# Task 4(d): Nested dictionary by style, sorted by count desc
style_dict = defaultdict(lambda: {"count": 0, "texts": []})

for r in paragraph_records:
    style_dict[r["style"]]["count"] += 1
    style_dict[r["style"]]["texts"].append(r["text"])

# Trim texts list to first 2 if more than 2 paragraphs share the style
for style, data in style_dict.items():
    if data["count"] > 2:
        data["texts"] = data["texts"][:2]

sorted_style_dict = dict(
    sorted(style_dict.items(), key=lambda item: item[1]["count"], reverse=True)
)

import pprint
pprint.pprint(sorted_style_dict)

{'Heading 1': {'count': 4, 'texts': ['Introduction', 'Getting Started']},
 'Heading 2': {'count': 1, 'texts': ['Team Structure']},
 'Normal': {'count': 10,
            'texts': ['Welcome to the team. This guide will walk you through '
                      'everything you need to know during your first few '
                      'weeks, including how to set up your accounts, meet your '
                      'colleagues, and understand our company culture.',
                      'Please read each section carefully and reach out to '
                      'your manager if you have any questions along the way.']},
 'Title': {'count': 1, 'texts': ['Company Onboarding Guide']}}


---
## Task 5: DOCX Table Extraction and Pandas Integration

In [20]:
# Task 5(a): Extract all tables as lists of lists
DOCX_TABLE_PATH = "sample_with_table.docx"
table_doc = Document(DOCX_TABLE_PATH)

all_tables = []
for table in table_doc.tables:
    table_data = []
    for row in table.rows:
        table_data.append([cell.text for cell in row.cells])
    all_tables.append(table_data)

print(f"Total number of tables found: {len(all_tables)}")
for i, t in enumerate(all_tables, start=1):
    rows = len(t)
    cols = len(t[0]) if rows > 0 else 0
    print(f"Table {i}: {rows} rows x {cols} columns")

Total number of tables found: 1
Table 1: 5 rows x 4 columns


In [21]:
# Task 5(b): First table -> DataFrame, count empty/None cells per column
first_table = all_tables[0]
header, data_rows = first_table[0], first_table[1:]

df_table = pd.DataFrame(data_rows, columns=header)
print(df_table.to_string())

empty_counts = {}
for col in df_table.columns:
    empty_counts[col] = df_table[col].apply(
        lambda v: v is None or str(v).strip() == ""
    ).sum()

print("\nEmpty/None cell counts per column:")
for col, count in empty_counts.items():
    print(f"{col}: {count}")

  Region Revenue Units Sold          Rep
0  North  125000        430   Alice Chen
1  South   98000             Brian Ortiz
2   East                310    Casey Kim
3   West  142000        512             

Empty/None cell counts per column:
Region: 0
Revenue: 1
Units Sold: 1
Rep: 1


In [22]:
# Task 5(c): Paragraphs before the first table
from docx.oxml.ns import qn

body_el = table_doc.element.body
pre_table_paragraphs = []

for child in body_el.iterchildren():
    if child.tag == qn('w:tbl'):
        break
    if child.tag == qn('w:p'):
        text = "".join(node.text or "" for node in child.iter(qn('w:t')))
        pre_table_paragraphs.append(text)

print(f"Paragraphs before first table: {len(pre_table_paragraphs)}")

Paragraphs before first table: 3


In [23]:
# Task 5(d): Write pre-table paragraphs + first table data to docx_extracted.txt
extracted_filename = "docx_extracted.txt"

with open(extracted_filename, "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para in pre_table_paragraphs:
        f.write(para + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print(f"Wrote extracted content to {extracted_filename}")
with open(extracted_filename, "r", encoding="utf-8") as f:
    print(f.read())

Wrote extracted content to docx_extracted.txt
--- PRE-TABLE PARAGRAPHS ---
Quarterly Sales Report
This report summarizes regional sales performance for the second quarter. Figures are broken down by region, including total revenue, number of units sold, and the assigned sales representative.
All figures are provided in US dollars and reflect finalized, audited totals as of the end of the reporting period.

--- TABLE DATA ---
Region | Revenue | Units Sold | Rep
North | 125000 | 430 | Alice Chen
South | 98000 |  | Brian Ortiz
East |  | 310 | Casey Kim
West | 142000 | 512 | 



---
## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

In [24]:
# Task 6(a): Heading comparison between two documents
DOC_A_PATH = "document_a.docx"
DOC_B_PATH = "document_b.docx"

doc_a = Document(DOC_A_PATH)
doc_b = Document(DOC_B_PATH)

headings_a = {p.text.strip() for p in doc_a.paragraphs if p.style.name.startswith("Heading")}
headings_b = {p.text.strip() for p in doc_b.paragraphs if p.style.name.startswith("Heading")}

common_headings = headings_a & headings_b
only_in_a = headings_a - headings_b
only_in_b = headings_b - headings_a

print(f"Headings common to both: {common_headings}")
print(f"Headings only in Document A: {only_in_a}")
print(f"Headings only in Document B: {only_in_b}")

Headings common to both: {'Budget Considerations', 'Project Overview'}
Headings only in Document A: {'Timeline and Milestones', 'Risks and Mitigations'}
Headings only in Document B: {'Support and Training', 'Mobile Rollout Plan'}


In [25]:
# Task 6(b): Normal paragraphs merge + dedupe
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal"]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal"]

merged_normal = normal_a + normal_b
count_before = len(merged_normal)

seen = set()
deduped_normal = []
for text in merged_normal:
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        deduped_normal.append(text)

count_after = len(deduped_normal)

print(f"Total paragraphs before deduplication: {count_before}")
print(f"Total paragraphs after deduplication: {count_after}")

Total paragraphs before deduplication: 10
Total paragraphs after deduplication: 9


In [26]:
# Task 6(c): Combined word frequency (top 20), 15+ stopwords removed
stopwords_15plus = BASIC_STOPWORDS | {
    'i','you','he','she','we','they','them','him','her','if','so','than'
}
assert len(stopwords_15plus) >= 15

all_tokens_combined = []
for text in deduped_normal:
    cleaned_text = re.sub(f"[{re.escape(string.punctuation)}]", '', text).lower()
    all_tokens_combined.extend(cleaned_text.split())

combined_freq = Counter(t for t in all_tokens_combined if t not in stopwords_15plus)

print("Top 20 most frequent words:")
for word, count in combined_freq.most_common(20):
    print(f"{word}: {count}")

Top 20 most frequent words:
project: 4
dashboard: 4
mobile: 4
key: 3
metrics: 3
phase: 3
managers: 3
internal: 2
time: 2
across: 2
six: 2
months: 2
milestone: 2
focused: 2
data: 2
migration: 2
second: 2
two: 2
thousand: 2
dollars: 2


In [27]:
# Task 6(d): Summary statistics
total_headings = len(headings_a) + len(headings_b)
total_unique_body = len(deduped_normal)
vocabulary_size = len(set(all_tokens_combined))

word_lengths = [len(text.split()) for text in deduped_normal]
avg_paragraph_len = round(sum(word_lengths) / len(word_lengths), 2) if word_lengths else 0

longest_paragraph = max(deduped_normal, key=lambda t: len(t.split())) if deduped_normal else ""

print(f"Total headings across both documents: {total_headings}")
print(f"Total unique body paragraphs after dedup: {total_unique_body}")
print(f"Total unique word types (vocabulary size): {vocabulary_size}")
print(f"Average paragraph length (words): {avg_paragraph_len}")
print(f"Longest paragraph by word count: {longest_paragraph}")

Total headings across both documents: 8
Total unique body paragraphs after dedup: 9
Total unique word types (vocabulary size): 140
Average paragraph length (words): 25.0
Longest paragraph by word count: This project aims to modernize our internal reporting tools by replacing legacy spreadsheets with a centralized dashboard. The new system will provide real time visibility into key performance metrics across every department.


---
## Task 7: Multi-Endpoint API Data Joining and Analysis

In [28]:
# Task 7(a): Fetch users and posts, validate status codes
USERS_URL = "https://jsonplaceholder.typicode.com/users"
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"

def get_json_or_raise(url):
    resp = requests.get(url)
    if resp.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {resp.status_code}")
    return resp.json()

users = get_json_or_raise(USERS_URL)
posts = get_json_or_raise(POSTS_URL)

print(f"Fetched {len(users)} users and {len(posts)} posts.")

Fetched 10 users and 100 posts.


In [29]:
# Task 7(b): Join users and posts
joined = {}
for user in users:
    user_posts = [p["title"] for p in posts if p["userId"] == user["id"]]
    joined[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": user_posts
    }

for name in list(joined.keys())[:3]:
    print(name, "->", joined[name])

Leanne Graham -> {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum']}
Ervin Howell -> {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem', 'dolorum ut in voluptas mollitia et saepe quo animi', 'voluptatem eligendi optio', 'eveniet quod temporibus', 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', 'fugit voluptas sed molestias voluptatem provident', 'voluptate et itaque vero tempora molestiae', 'adipisci placeat illum aut reiciendis qui', 'doloribus ad provident suscipit at']}
Clementine Bauch -> {'email

In [30]:
# Task 7(c): User with the most posts
users_by_id = {u["id"]: u for u in users}
name_to_id = {u["name"]: u["id"] for u in users}

top_user_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user = users_by_id[name_to_id[top_user_name]]

print(f"Full name: {top_user['name']}")
print(f"Email: {top_user['email']}")
print(f"Company: {top_user['company']['name']}")
print("Post titles:")
for i, title in enumerate(joined[top_user_name]["titles"], start=1):
    print(f"{i}. {title}")

Full name: Leanne Graham
Email: Sincere@april.biz
Company: Romaguera-Crona
Post titles:
1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2. qui est esse
3. ea molestias quasi exercitationem repellat qui ipsa sit aut
4. eum et est occaecati
5. nesciunt quas odio
6. dolorem eum magni eos aperiam quia
7. magnam facilis autem
8. dolorem dolore est ipsam
9. nesciunt iure omnis dolorem tempora et accusantium
10. optio molestias id quia eum


In [31]:
# Task 7(d): Filter users by company.bs 'synergies' or city length > 8
matching_users = []
for user in users:
    bs_match = "synergies" in user["company"]["bs"].lower()
    city_match = len(user["address"]["city"]) > 8
    if bs_match or city_match:
        matching_users.append(user)

for user in matching_users:
    post_count = joined[user["name"]]["post_count"]
    print(f"Name: {user['name']}, City: {user['address']['city']}, Post count: {post_count}")

Name: Leanne Graham, City: Gwenborough, Post count: 10
Name: Ervin Howell, City: Wisokyburgh, Post count: 10
Name: Clementine Bauch, City: McKenziehaven, Post count: 10
Name: Patricia Lebsack, City: South Elvis, Post count: 10
Name: Chelsey Dietrich, City: Roscoeview, Post count: 10
Name: Mrs. Dennis Schulist, City: South Christy, Post count: 10
Name: Kurtis Weissnat, City: Howemouth, Post count: 10
Name: Nicholas Runolfsdottir V, City: Aliyaview, Post count: 10
Name: Glenna Reichert, City: Bartholomebury, Post count: 10
Name: Clementina DuBuque, City: Lebsackbury, Post count: 10


---
## Task 8: To-Do Completion Rate Analysis Across Users

In [32]:
# Task 8(a): Fetch and validate todos
TODOS_URL = "https://jsonplaceholder.typicode.com/todos"

resp = requests.get(TODOS_URL)
if resp.status_code != 200:
    print(f"Error: request failed with status code {resp.status_code}")
    raise SystemExit

todos = resp.json()

if not isinstance(todos, list):
    print("Error: response is not a list.")
    raise SystemExit

if len(todos) < 1:
    print("Error: no todo records found.")
    raise SystemExit

print(f"Total number of todo records retrieved: {len(todos)}")

Total number of todo records retrieved: 200


In [33]:
# Task 8(b): Group todos by userId
todo_stats = defaultdict(lambda: {"total": 0, "completed": 0, "incomplete": 0})

for todo in todos:
    uid = todo["userId"]
    todo_stats[uid]["total"] += 1
    if todo["completed"]:
        todo_stats[uid]["completed"] += 1
    else:
        todo_stats[uid]["incomplete"] += 1

for uid in sorted(todo_stats):
    stats = todo_stats[uid]
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats}  | completed+incomplete==total: {check}")

User 1: {'total': 20, 'completed': 11, 'incomplete': 9}  | completed+incomplete==total: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12}  | completed+incomplete==total: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13}  | completed+incomplete==total: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14}  | completed+incomplete==total: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8}  | completed+incomplete==total: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14}  | completed+incomplete==total: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11}  | completed+incomplete==total: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9}  | completed+incomplete==total: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12}  | completed+incomplete==total: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8}  | completed+incomplete==total: True


In [34]:
# Task 8(c): Completion rates
completion_rates = {}
for uid, stats in todo_stats.items():
    rate = round((stats["completed"] / stats["total"]) * 100, 1) if stats["total"] else 0.0
    completion_rates[uid] = rate

highest_uid = max(completion_rates, key=completion_rates.get)
lowest_uid = min(completion_rates, key=completion_rates.get)
overall_avg = round(sum(completion_rates.values()) / len(completion_rates), 1)

print(f"Highest completion rate: User {highest_uid} -> {completion_rates[highest_uid]}%")
print(f"Lowest completion rate: User {lowest_uid} -> {completion_rates[lowest_uid]}%")
print(f"Overall average completion rate: {overall_avg}%")

Highest completion rate: User 5 -> 60.0%
Lowest completion rate: User 4 -> 30.0%
Overall average completion rate: 45.0%


In [35]:
# Task 8(d): Formatted ranking table
users_resp = requests.get(USERS_URL)
users_for_lookup = users_resp.json()
user_lookup = {u["id"]: u["name"] for u in users_for_lookup}

ranking = sorted(completion_rates.items(), key=lambda item: item[1], reverse=True)

header = f"{'Rank':<6}{'Name':<25}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %':<20}"
print(header)
print("-" * len(header))

for rank, (uid, rate) in enumerate(ranking, start=1):
    name = user_lookup.get(uid, f"User {uid}")
    total = todo_stats[uid]["total"]
    completed = todo_stats[uid]["completed"]
    print(f"{rank:<6}{name:<25}{total:<14}{completed:<12}{rate:<20}")

Rank  Name                     Total Tasks   Completed   Completion Rate %   
-----------------------------------------------------------------------------
1     Chelsey Dietrich         20            12          60.0                
2     Clementina DuBuque       20            12          60.0                
3     Leanne Graham            20            11          55.0                
4     Nicholas Runolfsdottir V 20            11          55.0                
5     Kurtis Weissnat          20            9           45.0                
6     Ervin Howell             20            8           40.0                
7     Glenna Reichert          20            8           40.0                
8     Clementine Bauch         20            7           35.0                
9     Patricia Lebsack         20            6           30.0                
10    Mrs. Dennis Schulist     20            6           30.0                


---
## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [36]:
# Task 9(a): Flatten nested user JSON
users_resp = requests.get(USERS_URL)
users_full = users_resp.json()

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

flattened_users = [flatten_user(u) for u in users_full]
for fu in flattened_users:
    print(fu)

{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
{'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.4572', 'lng': '-164.2990', '

In [37]:
# Task 9(b): DataFrame ops - cast, distance, farthest user
df_users = pd.DataFrame(flattened_users)
pd.set_option("display.max_columns", None)
print(df_users)

df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)

df_users["distance_from_origin"] = df_users.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1
)

farthest_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
print("\nUser with greatest distance from origin:")
print(farthest_user)

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

             city     zipcode       lat        lng        company_name  \
0     Gwenborough  92998-3874  -37.3159    81

In [38]:
# Task 9(c): Hemisphere classification
northern = df_users[df_users["lat"] > 0]["name"].tolist()
southern = df_users[df_users["lat"] < 0]["name"].tolist()

print(f"Northern Hemisphere users: {northern}")
print(f"Southern Hemisphere users: {southern}")

north_avg_dist = df_users[df_users["lat"] > 0]["distance_from_origin"].mean()
south_avg_dist = df_users[df_users["lat"] < 0]["distance_from_origin"].mean()

north_avg_dist = north_avg_dist if not math.isnan(north_avg_dist) else 0
south_avg_dist = south_avg_dist if not math.isnan(south_avg_dist) else 0

higher_hemisphere = "Northern" if north_avg_dist > south_avg_dist else "Southern"
print(f"Northern avg distance: {north_avg_dist:.4f}, Southern avg distance: {south_avg_dist:.4f}")
print(f"Hemisphere with higher average distance from origin: {higher_hemisphere}")

Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']
Northern avg distance: 123.5833, Southern avg distance: 84.3162
Hemisphere with higher average distance from origin: Northern


In [39]:
# Task 9(d): Comment counts per post
posts_resp = requests.get(POSTS_URL)
posts_full = posts_resp.json()

COMMENTS_URL = "https://jsonplaceholder.typicode.com/comments"
comments_resp = requests.get(COMMENTS_URL)
comments_full = comments_resp.json()

comment_count_by_post = Counter(c["postId"] for c in comments_full)

post_comment_tuples = [
    (p["id"], p["title"][:40], comment_count_by_post.get(p["id"], 0))
    for p in posts_full
]
post_comment_tuples.sort(key=lambda t: t[2], reverse=True)

print("Top 5 most-commented posts:")
for t in post_comment_tuples[:5]:
    print(t)

print("\nBottom 5 least-commented posts:")
for t in post_comment_tuples[-5:]:
    print(t)

Top 5 most-commented posts:
(1, 'sunt aut facere repellat provident occae', 5)
(2, 'qui est esse', 5)
(3, 'ea molestias quasi exercitationem repell', 5)
(4, 'eum et est occaecati', 5)
(5, 'nesciunt quas odio', 5)

Bottom 5 least-commented posts:
(96, 'quaerat velit veniam amet cupiditate aut', 5)
(97, 'quas fugiat ut perspiciatis vero provide', 5)
(98, 'laboriosam dolor voluptates', 5)
(99, 'temporibus sit alias delectus eligendi p', 5)
(100, 'at nam consequatur ea labore ea harum', 5)
